<a href="https://colab.research.google.com/github/ulysis022219/AudioToText-WhisperX/blob/main/AudioToText_WhisperX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗣️ AudioToText — WhisperX

**Turn any audio or video into text, subtitles, or a translation — right in your browser.**

Powered by [WhisperX](https://github.com/m-bain/whisperX) (faster Whisper with word-level timestamps). Nothing to install on your computer — it runs free on Google Colab's GPU.

---

### ▶️ How to use

1. **Turn on the GPU:** menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Run each step **in order, top to bottom** — click the round **▶️** button on the left of every cell.
3. Wait for a ✅ before moving to the next step.

> First time here? Just follow Steps 1 → 2 → 3 → 4. Steps 2.5 and 5 are optional extras.


## 1️⃣ Install (run once)

Click **▶️** below to install the pinned Colab environment. This takes a few minutes the first time.

> ⚠️ **The runtime will restart by itself after package verification. This is expected.** Continue to **Step 2** once it reconnects.


In [ ]:
#@title { display-mode: "form" }
# Pinned for Colab CPython 3.13. Verification uses package metadata because
# importing a replaced Torch stack in the old kernel can crash before restart.
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

FLAG = "/tmp/deps_installed_v3"
EXPECTED = {
    "torch": "2.8.0+cu128",
    "torchaudio": "2.8.0+cu128",
    "torchvision": "0.23.0+cu128",
    "torchcodec": "0.7.0+cu128",
    "whisperx": "3.8.6",
    "openai-whisper": "20250625",
    "openai": "3.6.0",
    "deepl": "1.32.0",
    "numpy": "2.5.2",
    "ctranslate2": "4.8.1",
}

def sh(desc, args):
    print(f"\n⏳ {desc}...")
    started = time.time()
    result = subprocess.run(args, capture_output=True, text=True)
    if result.returncode:
        print(f"❌ FAILED: {desc}")
        print(result.stdout[-1500:])
        print("--- STDERR ---")
        print(result.stderr[-3000:])
        raise RuntimeError(f"Step failed: {desc}")
    print(f"✅ {desc} ({time.time() - started:.0f}s)")
    return result

def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None

def verify_environment():
    problems = []
    print("\n🔍 Verifying installed packages...")
    for package, expected in EXPECTED.items():
        actual = installed_version(package)
        if actual == expected:
            print(f"  ✅ {package} {actual}")
        else:
            problems.append(f"{package}: expected {expected}, found {actual or 'missing'}")
    if subprocess.run(["ffmpeg", "-version"], capture_output=True).returncode:
        problems.append("ffmpeg executable: missing")
    else:
        print("  ✅ ffmpeg executable")
    if problems:
        raise RuntimeError("Environment verification failed:\n  - " + "\n  - ".join(problems))

if not os.path.exists(FLAG):
    if subprocess.run(["ffmpeg", "-version"], capture_output=True).returncode:
        if sys.platform == "linux":
            sh("Installing ffmpeg", ["apt-get", "install", "-y", "ffmpeg"])
        else:
            raise RuntimeError("Install ffmpeg first: https://ffmpeg.org/download.html")
    else:
        print("✅ ffmpeg already present")

    sh("Installing uv", [sys.executable, "-m", "pip", "install", "-q", "uv==0.8.22"])
    uv = [sys.executable, "-m", "uv", "pip", "install", "--system"]

    torch_ok = all(installed_version(name) == EXPECTED[name]
                   for name in ("torch", "torchaudio", "torchvision"))
    if torch_ok:
        print("✅ Exact PyTorch cu128 stack already installed — skipping download")
    else:
        sh("Installing PyTorch 2.8.0 (cu128)", uv + [
            "torch==2.8.0+cu128", "torchaudio==2.8.0+cu128",
            "torchvision==0.23.0+cu128", "--index-url",
            "https://download.pytorch.org/whl/cu128",
        ])

    sh("Installing WhisperX and optional backends", uv + [
        "whisperx==3.8.6", "openai-whisper==20250625", "openai==3.6.0",
        "deepl==1.32.0",
        "numpy==2.5.2", "ctranslate2==4.8.1", "torchcodec==0.7.0+cu128",
        "--extra-index-url", "https://download.pytorch.org/whl/cu128",
        "--index-strategy", "unsafe-best-match",
    ])
    verify_environment()
    Path = __import__("pathlib").Path
    Path(FLAG).write_text("verified", encoding="utf-8")
    print("\n✅ Installation complete. Restarting runtime...")
    sys.stdout.flush()
    os.kill(os.getpid(), 9)
else:
    verify_environment()
    print("\n✅ Dependencies verified. Continue to Step 2.")


## 2️⃣ Add your audio files

Click **▶️** below to connect Google Drive. The notebook creates and lists:

```
📁 My Drive › for process
```

Copy a printed path into `audio_file` in Step 3. Files and later outputs remain in Drive between Colab sessions.


In [ ]:
#@title 📁 Connect Google Drive { display-mode: "form" }
#@markdown Run this cell, approve Drive access, then add audio/video files.
#@markdown Files remain in Drive between sessions.
drive_folder = "/content/drive/MyDrive/for process" #@param {type:"string"}

from google.colab import drive
import os

drive.mount("/content/drive")
os.makedirs(drive_folder, exist_ok=True)
print(f"\n✅ Drive connected. Put files here: {drive_folder}\n")
files = sorted(
    os.path.join(drive_folder, name)
    for name in os.listdir(drive_folder)
    if os.path.isfile(os.path.join(drive_folder, name))
)
if files:
    print("Files found (copy a path into Step 3):")
    for filename in files:
        print("  •", filename)
else:
    print("Folder is empty — add files in Drive or the Colab Files panel.")


## 2.5 🎙️ Record your own audio *(optional)*

Run the cell, allow microphone access, then stop the recording. It saves to your Drive input folder by default.

> Skip this step if you already added a file in Step 2.


In [ ]:
#@title 🎙️ Record audio { display-mode: "form" }
import base64
import json
import os
import subprocess
import tempfile
from google.colab.output import eval_js
from IPython.display import HTML, display

recording_file = "/content/drive/MyDrive/for process/recording.wav" #@param {type:"string"}
overwrite_existing = False #@param {type:"boolean"}

AUDIO_HTML = r"""
<button id="recordButton">🎙️ Start recording</button>
<span id="recordStatus"></span>
<script>
const button = document.getElementById('recordButton');
const status = document.getElementById('recordStatus');
let recorder, stream, chunks = [];
window.recordingData = new Promise((resolve, reject) => {
  button.onclick = async () => {
    if (!recorder || recorder.state === 'inactive') {
      try {
        stream = await navigator.mediaDevices.getUserMedia({audio: true});
        chunks = [];
        recorder = new MediaRecorder(stream, {mimeType: 'audio/webm;codecs=opus'});
        recorder.ondataavailable = event => chunks.push(event.data);
        recorder.onstop = () => {
          const reader = new FileReader();
          reader.onloadend = () => resolve(reader.result);
          reader.readAsDataURL(new Blob(chunks, {type: 'audio/webm'}));
        };
        recorder.start();
        button.textContent = '⏹️ Stop recording';
        status.textContent = ' Recording…';
      } catch (error) { reject(error.toString()); }
    } else {
      recorder.stop();
      stream.getTracks().forEach(track => track.stop());
      button.disabled = true;
      status.textContent = ' Saving…';
    }
  };
});
</script>
"""

if os.path.exists(recording_file) and not overwrite_existing:
    raise FileExistsError(f"{recording_file} already exists; rename it or enable overwrite_existing")
os.makedirs(os.path.dirname(os.path.abspath(recording_file)), exist_ok=True)
display(HTML(AUDIO_HTML))
data_url = eval_js("recordingData")
raw = base64.b64decode(data_url.split(",", 1)[1])
with tempfile.TemporaryDirectory() as temp_dir:
    webm_path = os.path.join(temp_dir, "recording.webm")
    wav_path = os.path.join(temp_dir, "recording.wav")
    with open(webm_path, "wb") as handle:
        handle.write(raw)
    subprocess.run(["ffmpeg", "-y", "-i", webm_path, wav_path],
                   check=True, capture_output=True)
    os.replace(wav_path, recording_file)
eval_js(f"document.getElementById('recordStatus').textContent = {json.dumps(' Saved: ' + recording_file)}")
print(f"✅ Saved: {recording_file}")


## 3️⃣ 🎬 Transcribe or translate

1. Choose the **local WhisperX** backend (audio stays in Colab) or explicitly choose **OpenAI API** (audio is uploaded to OpenAI).
2. Paste one file path per line. Newline input safely supports filenames containing commas.
3. Click **▶️**. Progress is shown while each file runs; segments print when transcription completes.

Raw results are checkpointed to Google Drive after every completed file, so Step 4 can recover after a runtime interruption. The pinned WhisperX VAD checkpoint is hash-verified before its narrowly scoped legacy load; forced weights-only loading remains enabled for alignment models.


In [ ]:
#@title 🎬 Step 3 — Transcribe / Translate { display-mode: "form" }
import gc
import hashlib
import json
import logging
import os
os.environ["TORCH_FORCE_WEIGHTS_ONLY_LOAD"] = "1"
import re
import subprocess
import tempfile
import warnings
from pathlib import Path

#@markdown ### Backend and privacy
#@markdown Local WhisperX keeps audio in Colab. OpenAI API uploads audio to OpenAI.
transcription_backend = "Local WhisperX (Colab GPU)" #@param ["Local WhisperX (Colab GPU)", "OpenAI API (uploads audio)"]
#@markdown ### Task
task_label = "Transcribe" #@param ["Transcribe", "Translate to English"]
#@markdown ### Content profile
#@markdown ASMR tunes the ASR for heavy non-speech / mouth-sound audio: it disables
#@markdown previous-text conditioning (stops repetition on moaning), raises quality,
#@markdown locks Japanese when auto-detecting, auto-applies ASMR terminology, and marks
#@markdown non-speech as [ASMR sounds]. Recommended path: Transcribe here, then Step 5.
use_asmr_profile = True #@param {type:"boolean"}
task = "transcribe" if task_label == "Transcribe" else "translate"
#@markdown ### Input paths — one path per line
#@markdown Commas are allowed in filenames.
audio_file = "" #@param {type:"string"}
#@markdown ### Language and quality
language = "Auto-Detect" #@param ["Auto-Detect", "Afrikaans", "Albanian", "Amharic", "Arabic", "Armenian", "Assamese", "Azerbaijani", "Bashkir", "Basque", "Belarusian", "Bengali", "Bosnian", "Breton", "Bulgarian", "Burmese", "Castilian", "Catalan", "Chinese", "Croatian", "Czech", "Danish", "Dutch", "English", "Estonian", "Faroese", "Finnish", "Flemish", "French", "Galician", "Georgian", "German", "Greek", "Gujarati", "Haitian", "Haitian Creole", "Hausa", "Hawaiian", "Hebrew", "Hindi", "Hungarian", "Icelandic", "Indonesian", "Italian", "Japanese", "Javanese", "Kannada", "Kazakh", "Khmer", "Korean", "Lao", "Latin", "Latvian", "Letzeburgesch", "Lingala", "Lithuanian", "Luxembourgish", "Macedonian", "Malagasy", "Malay", "Malayalam", "Maltese", "Maori", "Marathi", "Moldavian", "Moldovan", "Mongolian", "Myanmar", "Nepali", "Norwegian", "Nynorsk", "Occitan", "Panjabi", "Pashto", "Persian", "Polish", "Portuguese", "Punjabi", "Pushto", "Romanian", "Russian", "Sanskrit", "Serbian", "Shona", "Sindhi", "Sinhala", "Sinhalese", "Slovak", "Slovenian", "Somali", "Spanish", "Sundanese", "Swahili", "Swedish", "Tagalog", "Tajik", "Tamil", "Tatar", "Telugu", "Thai", "Tibetan", "Turkish", "Turkmen", "Ukrainian", "Urdu", "Uzbek", "Valencian", "Vietnamese", "Welsh", "Yiddish", "Yoruba"]
use_model = "large-v3" #@param ["tiny", "base", "small", "medium", "large-v1", "large-v2", "large-v3"]
quality_mode = "Balanced" #@param ["Balanced", "High accuracy"]
context_conditioning = False #@param {type:"boolean"}
#@markdown ### Terminology hints for names, acronyms, terms and subject area (optional)
#@markdown Hints improve proper nouns and specialist vocabulary.
prompt_names = "" #@param {type:"string"}
prompt_terms = "" #@param {type:"string"}
prompt_acronyms = "" #@param {type:"string"}
subject_area = "" #@param {type:"string"}
prompt = "" #@param {type:"string"}
#@markdown ### Advanced
compute_type = "float16" #@param ["float16", "int8"]
batch_size = 8 #@param {type:"integer"}
word_timestamps = True #@param {type:"boolean"}
fail_if_alignment_fails = False #@param {type:"boolean"}
max_chars_per_line = 20 #@param {type:"integer"}

if not audio_file.strip():
    raise ValueError("audio_file is empty — paste one path per line from Step 2")
audio_files = [line.strip() for line in audio_file.splitlines() if line.strip()]
for audio_path in audio_files:
    if not os.path.isfile(audio_path):
        raise FileNotFoundError(f"File not found: {audio_path}")
if batch_size < 1 or max_chars_per_line < 1:
    raise ValueError("batch_size and max_chars_per_line must be positive")

# Combine free-text and structured hints into one concise prompt.
structured_hint_parts = []
if subject_area.strip():
    structured_hint_parts.append(subject_area.strip())
if prompt_names.strip():
    structured_hint_parts.append("Names: " + prompt_names.strip())
if prompt_terms.strip():
    structured_hint_parts.append("Terms: " + prompt_terms.strip())
if prompt_acronyms.strip():
    structured_hint_parts.append("Acronyms: " + prompt_acronyms.strip())
effective_prompt = " ".join(filter(None, [prompt.strip(), " ".join(structured_hint_parts)])).strip()

quality_presets = {
    "Balanced": {"beam_size": 3, "best_of": 3},
    "High accuracy": {"beam_size": 5, "best_of": 5},
}
if quality_mode not in quality_presets:
    raise ValueError(f"Unknown quality mode: {quality_mode}")
quality = quality_presets[quality_mode]

# --- ASMR content profile ------------------------------------------------
# When ON, override a General-purpose preset with settings tuned for
# ear-licking / mouth-sound / moaning-heavy audio. Manual knobs above still
# work; this just applies sensible ASMR defaults.

def mark_non_speech(segments, no_speech_threshold=0.6, placeholder="[ASMR sounds]"):
    """Replace whisper's hallucinated chatter on non-speech with a placeholder."""
    output = []
    for segment in segments:
        text = str(segment.get("text", "")).strip()
        if (text and segment.get("no_speech_prob") is not None
                and segment["no_speech_prob"] > no_speech_threshold):
            segment = {**segment, "text": placeholder}
        output.append(segment)
    return output

_ASMR_PROFILE_ACTIVE = False
if use_asmr_profile:
    _ASMR_PROFILE_ACTIVE = True
    # Repetition on moaning is the classic ASMR failure; previous-text
    # conditioning makes it worse, so force it off regardless of the checkbox.
    context_conditioning = False
    quality_mode = "High accuracy"
    quality = quality_presets[quality_mode]
    if language == "Auto-Detect":
        # This corpus is Japanese; lock it so the language does not flip-flop.
        language = "Japanese"

    # ASMR / ero glossary used as hotword hints on the ASR side.
    asmr_names = "Kanon, Kana-chan, Onii-san"
    asmr_terms = ("paizuri, ear licking, ear-licking, nipple, nipples, breast, "
                  "breasts, tongue, kiss, kissing, sloppy, chupa, chupo, peropero, "
                  "shikoshiko, nucho, gyu, guri guri, suko, omanko, penis, cock, cum, "
                  "squirt, lotion, oil, ero-goumon, onasapo, tildo, yawachichi")
    asmr_acronyms = "ASMR, R18, KU100, R15"
    asmr_subject = "Japanese ASMR ear-licking R18 audio roleplay"
    asmr_hint = " ".join([
        effective_prompt, asmr_subject,
        "Names: " + asmr_names, "Terms: " + asmr_terms,
        "Acronyms: " + asmr_acronyms,
    ]).strip()
    if asmr_hint:
        effective_prompt = asmr_hint
    print(" ASMR profile ON: quality=High accuracy, conditioning OFF, "
          "Japanese fixed, ASMR terms applied.")
else:
    print("Content profile: General (no ASMR tuning).")


from google.colab import userdata
import whisper as whisper_tokenizer
WHISPER_LANGUAGES = whisper_tokenizer.tokenizer.TO_LANGUAGE_CODE
lang_code = None if language == "Auto-Detect" else WHISPER_LANGUAGES.get(language.lower())
if language != "Auto-Detect" and not lang_code:
    raise ValueError(f"Unsupported language: {language}")

CHECKPOINT_PATH = "/content/drive/MyDrive/audio_transcription/_last_results.json"
CJK_LANGUAGE_CODES = {"ja", "zh"}
WHISPERX_VAD_SHA256 = "0b5b3216d60a2d32fc086b47ea8c67589aaeb26b7e07fcbe620d6d0b83e209ea"

def verify_file_sha256(path, expected_sha256):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != expected_sha256:
        raise RuntimeError(f"Security check failed for {path}: unexpected SHA-256")

def load_whisperx_with_verified_vad(model_name, device, **model_options):
    from whisperx.vads import pyannote as pyannote_vad
    vad_path = Path(pyannote_vad.__file__).resolve().parents[1] / "assets" / "pytorch_model.bin"
    verify_file_sha256(vad_path, WHISPERX_VAD_SHA256)

    # WhisperX 3.8.6 bundles this exact legacy checkpoint. Load only that
    # verified file outside the global weights-only policy, then restore it.
    force_weights_only = os.environ.pop("TORCH_FORCE_WEIGHTS_ONLY_LOAD", None)
    try:
        vad_model = pyannote_vad.Pyannote(
            torch.device(device), model_fp=str(vad_path), chunk_size=30,
            vad_onset=0.500, vad_offset=0.363)
    finally:
        if force_weights_only is not None:
            os.environ["TORCH_FORCE_WEIGHTS_ONLY_LOAD"] = force_weights_only

    return whisperx.load_model(
        model_name, device, vad_model=vad_model, **model_options)

def format_timestamp(seconds):
    milliseconds = max(0, round(float(seconds) * 1000))
    hours, milliseconds = divmod(milliseconds, 3_600_000)
    minutes, milliseconds = divmod(milliseconds, 60_000)
    secs, milliseconds = divmod(milliseconds, 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}.{milliseconds:03d}"

def atomic_json_dump(data, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temp_name = None
    try:
        with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=destination.parent,
                                         prefix=destination.name + ".", suffix=".tmp",
                                         delete=False) as handle:
            temp_name = handle.name
            json.dump(data, handle, ensure_ascii=False,
                      default=lambda value: float(value) if hasattr(value, "__float__") else str(value))
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temp_name, destination)
    except Exception:
        if temp_name:
            Path(temp_name).unlink(missing_ok=True)
        raise

def join_word_texts(words, cjk=False):
    pieces = [str(word.get("word", word.get("text", ""))).strip()
              for word in words]
    pieces = [piece for piece in pieces if piece]
    if cjk:
        return "".join(pieces)
    text = " ".join(pieces)
    return re.sub(r"\s+([,.;:!?])", r"\1", text)

def split_long_segments(segments, max_chars=20, language_code=None):
    if language_code not in CJK_LANGUAGE_CODES:
        return segments
    output = []
    for segment in segments:
        words = [word for word in segment.get("words", [])
                 if "start" in word and "end" in word]
        if not words:
            output.append(segment)
            continue
        chunk = []
        for word in words:
            chunk.append(word)
            if len(join_word_texts(chunk, cjk=True)) >= max_chars:
                output.append({**segment, "start": chunk[0]["start"],
                               "end": chunk[-1]["end"],
                               "text": join_word_texts(chunk, cjk=True),
                               "words": chunk})
                chunk = []
        if chunk:
            output.append({**segment, "start": chunk[0]["start"],
                           "end": chunk[-1]["end"],
                           "text": join_word_texts(chunk, cjk=True),
                           "words": chunk})
    return output

def collect_quality_notes(segments, thresholds=None):
    thresholds = thresholds or {"avg_logprob": -1.0, "no_speech_prob": 0.6, "compression_ratio": 2.0}
    notes = []
    for segment in segments:
        reasons = []
        avg_logprob = segment.get("avg_logprob")
        no_speech = segment.get("no_speech_prob")
        compression = segment.get("compression_ratio")
        if avg_logprob is not None and avg_logprob < thresholds["avg_logprob"]:
            reasons.append(f"low confidence ({avg_logprob:.2f})")
        if no_speech is not None and no_speech > thresholds["no_speech_prob"]:
            reasons.append(f"possible non-speech ({no_speech:.2f})")
        if compression is not None and compression > thresholds["compression_ratio"]:
            reasons.append(f"repetition risk ({compression:.2f})")
        if reasons:
            notes.append({"start": segment.get("start"), "end": segment.get("end"),
                          "text": segment.get("text", ""), "reasons": "; ".join(reasons)})
    return notes

def merge_api_segments(chunk_results):
    merged = []
    detected_language = None
    duration = 0.0
    for chunk in chunk_results:
        offset = float(chunk["offset_seconds"])
        response = chunk["response"]
        detected_language = detected_language or response.get("language")
        for original in response.get("segments", []):
            segment = dict(original)
            segment["start"] = float(segment.get("start", 0)) + offset
            segment["end"] = float(segment.get("end", 0)) + offset
            merged.append(segment)
            duration = max(duration, segment["end"])
    merged.sort(key=lambda segment: (segment["start"], segment["end"]))
    return {"segments": merged, "language": detected_language, "duration": duration}

def prepare_api_chunks(audio_path, temp_dir, max_bytes=24 * 1024 * 1024):
    supported = {"mp3", "mp4", "mpeg", "mpga", "m4a", "wav", "webm"}
    extension = Path(audio_path).suffix.lower().lstrip(".")
    if extension in supported and os.path.getsize(audio_path) < max_bytes:
        return [{"path": audio_path, "offset_seconds": 0.0}]

    probe = subprocess.run([
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", audio_path,
    ], check=True, capture_output=True, text=True)
    duration_seconds = float(probe.stdout.strip())
    # 64 kbps CBR plus 15% container/headroom; the size check below is authoritative.
    target_seconds = max(1.0, max_bytes * 8 / 64_000 * 0.85)
    chunks = []

    def export_piece(start_seconds, piece_seconds):
        path = os.path.join(temp_dir, f"chunk-{len(chunks):04d}.mp3")
        subprocess.run([
            "ffmpeg", "-y", "-ss", str(start_seconds), "-i", audio_path,
            "-t", str(piece_seconds), "-vn", "-ac", "1", "-ar", "16000",
            "-b:a", "64k", path,
        ], check=True, capture_output=True)
        if os.path.getsize(path) >= max_bytes:
            os.remove(path)
            if piece_seconds <= 1.0:
                raise RuntimeError("Unable to create an API chunk below the upload limit")
            first = piece_seconds / 2
            export_piece(start_seconds, first)
            export_piece(start_seconds + first, piece_seconds - first)
        else:
            chunks.append({"path": path, "offset_seconds": float(start_seconds)})

    start = 0.0
    while start < duration_seconds:
        piece_seconds = min(target_seconds, duration_seconds - start)
        export_piece(start, piece_seconds)
        start += piece_seconds
    if any(os.path.getsize(chunk["path"]) >= max_bytes for chunk in chunks):
        raise RuntimeError("API chunk verification failed")
    return chunks

def free_vram():
    gc.collect()
    if "torch" in globals() and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def is_cuda_out_of_memory(error):
    message = str(error).lower()
    return "out of memory" in message and ("cuda" in message or "cublas" in message)

def transcribe_with_batch_backoff(model, audio, initial_batch_size, **options):
    current_batch_size = initial_batch_size
    while True:
        try:
            return (model.transcribe(audio, batch_size=current_batch_size, **options),
                    current_batch_size)
        except RuntimeError as error:
            if not is_cuda_out_of_memory(error):
                raise
            if current_batch_size == 1:
                raise RuntimeError(
                    "CUDA ran out of memory even at batch_size=1. Restart the runtime, "
                    "then use a smaller model or compute_type='int8'.") from error
            next_batch_size = max(1, current_batch_size // 2)
            print(f"⚠️ CUDA out of memory at batch_size={current_batch_size}; "
                  f"retrying with batch_size={next_batch_size}…")
            current_batch_size = next_batch_size
        # Run after leaving the exception handler so its traceback no longer retains tensors.
        free_vram()

SAFE_ALIGN_MODELS_HF = {
    # Immutable revisions verified to contain safetensors weights.
    "ja": ("jonatasgrosman/wav2vec2-large-xlsr-53-japanese", "2785e99ab97df77a32b5bd0ece5c9fa188a02f19"),
    "cs": ("comodoro/wav2vec2-xls-r-300m-cs-250", "73e2b9004f35d3ca79316a624e7edc3cdaa45d40"),
    "da": ("saattrupdan/wav2vec2-xls-r-300m-ftspeech", "7a60985d596e7a9e26b01a0707fe59daac6ffe84"),
    "eu": ("stefan-it/wav2vec2-large-xlsr-53-basque", "ac4e27d2d376a5ed919b381ec99d7a86b37c0a6b"),
    "he": ("imvladikon/wav2vec2-xls-r-300m-hebrew", "b2e683004903f1b11bee2c6092a7c323ed858d82"),
    "hi": ("theainerd/Wav2Vec2-large-xlsr-hindi", "062f7f566e2671336992b011dcb9387cd3cffe5e"),
    "hr": ("classla/wav2vec2-xls-r-parlaspeech-hr", "a6075e0c9e0c49a533dc61876b1eafc6bc78de92"),
    "ko": ("kresnik/wav2vec2-large-xlsr-korean", "629c9a3501c10ba128bf3fa1eebb12af3be03f61"),
    "lv": ("jimregan/wav2vec2-large-xlsr-latvian-cv", "ca99957784deb8ffadf4b782db3e96dde797d164"),
    "nn": ("NbAiLab/nb-wav2vec2-1b-nynorsk", "37107327f6af742c008042d8cfdba4276de18f82"),
    "no": ("NbAiLab/nb-wav2vec2-1b-bokmaal-v2", "58a6eb5d7927c378f5d2785bc865756c9ae273ae"),
    "ro": ("gigant/romanian-wav2vec2", "79cf603aac59501d02bfeb37f615efc3ac4ce1b3"),
    "sv": ("KBLab/wav2vec2-large-voxrex-swedish", "ca70e31c06a2617bf7fe3b4fb5d387d2b19b2983"),
    "uk": ("Yehor/wav2vec2-xls-r-300m-uk-with-small-lm", "e3ced4def0d70be3aab0f2db598a59961fe9ab3b"),
    "ur": ("kingabzpro/wav2vec2-large-xls-r-300m-Urdu", "18a6595fc0cc4d9c63fc8d0407b5596f09c49edb"),
}

def load_safe_align_model(language_code, device):
    from whisperx.alignment import DEFAULT_ALIGN_MODELS_TORCH
    if language_code in DEFAULT_ALIGN_MODELS_TORCH:
        model_a, metadata = whisperx.load_align_model(language_code=language_code, device=device)
        return model_a, metadata, {
            "source": "torchaudio", "model": DEFAULT_ALIGN_MODELS_TORCH[language_code],
            "revision": "torchaudio==2.8.0+cu128", "weights_only": True}

    safe_model = SAFE_ALIGN_MODELS_HF.get(language_code)
    if not safe_model:
        raise RuntimeError(
            f"No pinned safetensors alignment model is available for language '{language_code}'; "
            "unsafe pickle fallback is disabled")
    model_id, revision = safe_model
    from huggingface_hub import snapshot_download
    safe_model_dir = snapshot_download(
        model_id, revision=revision,
        allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors"],
    )
    if not list(Path(safe_model_dir).glob("*.safetensors")):
        raise RuntimeError(f"Pinned alignment model {model_id}@{revision} did not provide safetensors")
    model_a, metadata = whisperx.load_align_model(
        language_code=language_code, device=device, model_name=safe_model_dir)
    return model_a, metadata, {
        "source": "huggingface", "model": model_id,
        "revision": revision, "weights": "safetensors"}

options = {
    "task": task,
    "hotwords": effective_prompt or None,
    "beam_size": quality["beam_size"],
    "best_of": quality["best_of"],
    "no_speech_threshold": 0.3,
    "compression_ratio_threshold": 2.6,
    "temperatures": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
}

use_openai = transcription_backend.startswith("OpenAI")
if use_openai:
    print("🔒 OpenAI mode selected: audio will be uploaded to OpenAI for processing.")
    try:
        openai_api_key = userdata.get("OPENAI_API_KEY") or ""
    except Exception:
        openai_api_key = ""
    if not openai_api_key:
        raise RuntimeError("Add OPENAI_API_KEY in Colab Secrets and enable notebook access")
    from openai import OpenAI
    api_client = OpenAI(api_key=openai_api_key)
else:
    print("🔒 Local mode selected: audio remains in this Colab runtime.")
    for logger_name in ("pytorch_lightning", "lightning_fabric", "pyannote", "speechbrain"):
        logging.getLogger(logger_name).setLevel(logging.ERROR)
    for warning_text in (".*TensorFloat-32.*", ".*was trained with.*", ".*does not match the version.*"):
        warnings.filterwarnings("ignore", message=warning_text)
    try:
        hf_token = userdata.get("HF_TOKEN") or ""
    except Exception:
        hf_token = ""
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("Hugging Face token loaded ✅")
    elif word_timestamps:
        print("ℹ️ No HF_TOKEN secret found. Public alignment models may still work; gated models will not.")

    import torch
    import whisperx
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cpu":
        compute_type = "int8"
        print("⚠️ GPU unavailable; using CPU/int8. This can be very slow.")
    else:
        print("Using GPU")
        subprocess.run(["nvidia-smi", "-L"], check=False)
        print(f"torch CUDA devices: {torch.cuda.device_count()} | "
              f"{torch.cuda.get_device_name(0) if torch.cuda.device_count() else 'none'}")
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    model_name = use_model + ".en" if language == "English" and use_model in {"tiny", "base", "small", "medium"} else use_model
    model_signature = (model_name, DEVICE, compute_type, lang_code, options["hotwords"])
    if "model" in globals() and globals().get("_model_signature") == model_signature:
        print(f"Reusing {model_name} already in memory ⚡")
    else:
        if "model" in globals():
            del model
        free_vram()
        print(f"Loading {model_name} with WhisperX…")
        model = load_whisperx_with_verified_vad(
            model_name, DEVICE, compute_type=compute_type, language=lang_code,
            asr_options={
                "condition_on_previous_text": context_conditioning,
                "no_speech_threshold": options["no_speech_threshold"],
                "compression_ratio_threshold": options["compression_ratio_threshold"],
                "temperatures": options["temperatures"],
                "beam_size": options["beam_size"],
                "best_of": options["best_of"],
                "hotwords": options["hotwords"],
            },
        )
        _model_signature = model_signature

results = {}
alignment_failures = []
print("-- TRANSLATE TO ENGLISH --" if task == "translate" else "-- TRANSCRIPTION --")

for audio_path in audio_files:
    print(f"\nProcessing: {audio_path}")
    if use_openai:
        api_options = {"response_format": "verbose_json"}
        if effective_prompt:
            api_options["prompt"] = effective_prompt
        if lang_code and task == "transcribe":
            api_options["language"] = lang_code
        endpoint = (api_client.audio.transcriptions if task == "transcribe"
                    else api_client.audio.translations).create
        chunk_results = []
        with tempfile.TemporaryDirectory(prefix="audio-to-text-") as temp_dir:
            chunks = prepare_api_chunks(audio_path, temp_dir)
            for index, chunk in enumerate(chunks, 1):
                print(f"Uploading chunk {index}/{len(chunks)} ({chunk['offset_seconds']:.1f}s offset)…")
                with open(chunk["path"], "rb") as audio_handle:
                    response = endpoint(model="whisper-1", file=audio_handle, **api_options).model_dump()
                chunk_results.append({"offset_seconds": chunk["offset_seconds"], "response": response})
        result = merge_api_segments(chunk_results)
        if task == "translate":
            result["source_language"] = result.get("language")
            result["language"] = "en"
        result["alignment_status"] = "not_available_for_openai_backend"
    else:
        audio = whisperx.load_audio(audio_path)
        duration_minutes = len(audio) / 16000 / 60
        print(f"Transcribing ~{duration_minutes:.1f} min (batch_size={batch_size})…")
        try:
            result, used_batch_size = transcribe_with_batch_backoff(
                model, audio, batch_size, language=lang_code, task=task,
                print_progress=False)
            if used_batch_size != batch_size:
                print(f"Continuing with batch_size={used_batch_size}.")
                batch_size = used_batch_size
        except RuntimeError as error:
            if "cudaErrorInvalidDevice" in str(error) or "invalid device ordinal" in str(error):
                raise RuntimeError(
                    "The GPU/device is invalid (cudaErrorInvalidDevice). Colab may have "
                    "reclaimed or reset the GPU, or the cached model is bound to a dead "
                    "device. Fix: Runtime \u2192 Restart runtime, then re-run from Step 1. If it "
                    "persists, Runtime \u2192 Change runtime type to a different GPU (or Factory "
                    "reset the runtime). You can also try compute_type='int8'."
                ) from error
            raise
        detected_language = result.get("language", lang_code)
        result["alignment_status"] = "not_requested"
        if word_timestamps and task == "transcribe":
            try:
                print("Aligning word timestamps…")
                model_a, metadata, alignment_model = load_safe_align_model(
                    detected_language, DEVICE)
                result = whisperx.align(result["segments"], model_a, metadata,
                                        audio, DEVICE, return_char_alignments=False)
                result["language"] = detected_language
                result["alignment_status"] = "complete"
                result["alignment_model"] = alignment_model
                del model_a, metadata
                free_vram()
            except Exception as error:
                result["alignment_status"] = "failed"
                result["alignment_error"] = str(error)
                alignment_failures.append(f"{audio_path}: {error}")
                print(f"⚠️ Word alignment failed: {error}")
                if fail_if_alignment_fails:
                    raise RuntimeError(f"Required alignment failed for {audio_path}: {error}") from error
        elif task == "translate":
            result["alignment_status"] = "not_applicable_to_translation"
        result["segments"] = split_long_segments(
            result["segments"], max_chars=max_chars_per_line,
            language_code=detected_language)
        result.setdefault("language", detected_language)
        result["quality_notes"] = collect_quality_notes(result["segments"])
        if task == "translate":
            result["source_language"] = detected_language
            result["language"] = "en"
        del audio
        free_vram()

    if use_asmr_profile:
        result["segments"] = mark_non_speech(result.get("segments", []))
    for segment in result.get("segments", []):
        segment["text"] = segment.get("text", "").strip()
        print(f"[{format_timestamp(segment.get('start', 0))} --> {format_timestamp(segment.get('end', 0))}] {segment['text']}")
    result["text"] = "\n".join(segment["text"] for segment in result.get("segments", []))
    results[audio_path] = result
    if result.get("quality_notes"):
        print(f"  ⚠️ Review {len(result['quality_notes'])} low-confidence segment(s): {audio_path}")
    checkpoint = {
        "schema_version": 1,
        "task": task,
        "backend": transcription_backend,
        "inputs": audio_files,
        "results": results,
        "asmr_profile": use_asmr_profile,
    }
    atomic_json_dump(checkpoint, CHECKPOINT_PATH)
    print(f"💾 Checkpoint updated: {CHECKPOINT_PATH}")

print(f"\n✅ Done. Processed {len(results)} file(s).")
if alignment_failures:
    print("⚠️ Completed with alignment degradation:")
    for failure in alignment_failures:
        print("  •", failure)
print("Run Step 4 to write outputs.")


## 4️⃣ 💾 Save your results

Outputs are saved persistently to **Google Drive** by default. If the runtime lost its in-memory state, this step validates and loads the last Step 3 checkpoint automatically.

Repetition cleanup is optional because repetition can be legitimate speech. Enabling raw JSON preserves the untouched word-level data beside cleaned outputs.


In [ ]:
#@title 💾 Step 4 — Save results { display-mode: "form" }
import copy
import hashlib
import json
import os
import re
import tempfile
from pathlib import Path
from typing import TextIO

output_dir = "/content/drive/MyDrive/audio_transcription" #@param {type:"string"}
output_formats = "srt" #@param ["txt,vtt,srt,tsv,json", "txt,vtt,srt", "txt,srt", "txt", "vtt", "srt", "tsv", "json"] {allow-input: true}
cleanup_repetitions = False #@param {type:"boolean"}
save_raw_json = True #@param {type:"boolean"}
overwrite_existing = False #@param {type:"boolean"}

ALLOWED_OUTPUT_FORMATS = {"txt", "vtt", "srt", "tsv", "json"}
CHECKPOINT_PATH = "/content/drive/MyDrive/audio_transcription/_last_results.json"

def validate_output_formats(value):
    formats = [item.strip().lower() for item in value.split(",") if item.strip()]
    invalid = sorted(set(formats) - ALLOWED_OUTPUT_FORMATS)
    if not formats or invalid:
        raise ValueError(f"Output formats must be from {sorted(ALLOWED_OUTPUT_FORMATS)}; invalid: {invalid}")
    return list(dict.fromkeys(formats))

def load_results_checkpoint(path):
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    if payload.get("schema_version") != 1 or not isinstance(payload.get("results"), dict):
        raise ValueError("Unsupported or invalid results checkpoint")
    return payload

def clean_repeated_words(text, threshold=4):
    if not text:
        return text
    text = re.sub(r"(.{1,10}?)\1{3,}", r"\1", text)
    text = re.sub(r"\b(\w+)(\s+\1){" + str(threshold - 1) + r",}\b",
                  r"\1", text, flags=re.IGNORECASE)
    text = re.sub(r"(.{8,60}?)(\s+\1){" + str(threshold - 2) + r",}",
                  r"\1", text, flags=re.IGNORECASE)
    text = re.sub(r"(\.\.\.)(\s*\.\.\.)+", r"\1", text)
    return re.sub(r"\s{2,}", " ", text).strip()

def clean_segments(segments):
    cleaned = []
    seen = []
    for segment in segments:
        text = clean_repeated_words(segment.get("text", "").strip())
        duplicate = any(text and previous and (
            text == previous or
            (len(text) > 10 and text in previous) or
            (len(previous) > 10 and previous in text)
        ) for previous in seen[-2:])
        if text and not duplicate:
            seen.append(text)
            cleaned.append({**segment, "text": text})
    return cleaned

def output_name_map(paths):
    stems = [Path(path).stem for path in paths]
    counts = {stem: stems.count(stem) for stem in set(stems)}
    return {
        path: (stem if counts[stem] == 1 else
               f"{stem}-{hashlib.sha256(path.encode('utf-8')).hexdigest()[:8]}")
        for path, stem in zip(paths, stems)
    }

def reserve_output_name(base_name, formats, directory, overwrite=False):
    if overwrite:
        return base_name
    suffix = 0
    while True:
        candidate = base_name if suffix == 0 else f"{base_name}-{suffix}"
        reserved = []
        try:
            for extension in formats:
                destination = os.path.join(directory, f"{candidate}.{extension}")
                descriptor = os.open(destination, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
                os.close(descriptor)
                reserved.append(destination)
            return candidate
        except FileExistsError:
            for destination in reserved:
                os.unlink(destination)
            suffix += 1
        except Exception:
            for destination in reserved:
                os.unlink(destination)
            raise

try:
    results
except NameError:
    results = None
if not results:
    if not os.path.isfile(CHECKPOINT_PATH):
        raise RuntimeError("No in-memory results or checkpoint found. Run Step 3 first.")
    checkpoint_payload = load_results_checkpoint(CHECKPOINT_PATH)
    results = checkpoint_payload["results"]
    task = checkpoint_payload.get("task", "transcribe")
    print(f"♻️ Recovered {len(results)} file(s) from {CHECKPOINT_PATH}")
    _ASMR_PROFILE_ACTIVE = bool(checkpoint_payload.get("asmr_profile", False))

# ASMR profile coordination: when Step 3 ran with the ASMR profile, default
# repetition cleanup ON (mouth-sound audio repeats heavily).
if globals().get("_ASMR_PROFILE_ACTIVE", False):
    cleanup_repetitions = True
    print(" ASMR profile active: repetition cleanup enabled.")

output_formats = validate_output_formats(output_formats)
os.makedirs(output_dir, exist_ok=True)
from whisper.utils import get_writer, WriteTXT

class WriteText(WriteTXT):
    def write_result(self, result: dict, file: TextIO, **kwargs):
        print(result["text"], file=file, flush=True)

def write_result(result, output_format, output_file_name, raw=False):
    safe_result = copy.deepcopy(result)
    if cleanup_repetitions and not raw:
        safe_result["segments"] = clean_segments(safe_result.get("segments", []))
        safe_result["text"] = "\n".join(segment["text"] for segment in safe_result["segments"])

    destination = os.path.join(output_dir, f"{output_file_name}.{output_format}")
    if output_format == "json":
        temp_path = None
        try:
            with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=output_dir,
                                             suffix=".tmp", delete=False) as handle:
                temp_path = handle.name
                json.dump(safe_result, handle, ensure_ascii=False, indent=2)
                handle.flush()
                os.fsync(handle.fileno())
            os.replace(temp_path, destination)
        except Exception:
            if temp_path:
                Path(temp_path).unlink(missing_ok=True)
            raise
        return destination

    for segment in safe_result.get("segments", []):
        segment.pop("words", None)
    writer_options = {"max_line_width": None, "max_line_count": None,
                      "highlight_words": False}
    with tempfile.TemporaryDirectory(dir=output_dir) as temp_dir:
        writer = WriteText(temp_dir) if output_format == "txt" else get_writer(output_format, temp_dir)
        writer(safe_result, output_file_name, writer_options)
        generated = os.path.join(temp_dir, f"{output_file_name}.{output_format}")
        os.replace(generated, destination)
    return destination

def normalize_subtitle(path):
    path = Path(path)
    raw = path.read_bytes()
    if raw.startswith(b"\xef\xbb\xbf"):
        raw = raw[3:]
    raw = raw.replace(b"\r\n", b"\n").replace(b"\r", b"\n")
    normalized = b"\xef\xbb\xbf" + raw.replace(b"\n", b"\r\n")
    with tempfile.NamedTemporaryFile("wb", dir=path.parent, suffix=".tmp", delete=False) as handle:
        handle.write(normalized)
        temporary = handle.name
    os.replace(temporary, path)

name_map = output_name_map(list(results))
generated_paths = []
for audio_path, result in results.items():
    reserved_formats = list(output_formats)
    if save_raw_json:
        reserved_formats.append("raw.json")
    base_name = reserve_output_name(name_map[audio_path], reserved_formats, output_dir,
                                    overwrite=overwrite_existing)
    reservations = [os.path.join(output_dir, f"{base_name}.{extension}")
                    for extension in reserved_formats]
    try:
        for output_format in output_formats:
            path = write_result(result, output_format, base_name)
            generated_paths.append(path)
            print("✅", path)
        if save_raw_json:
            raw_path = write_result(result, "json", base_name + ".raw", raw=True)
            generated_paths.append(raw_path)
            print("✅", raw_path)
    except Exception:
        for reservation in reservations:
            if reservation not in generated_paths:
                Path(reservation).unlink(missing_ok=True)
        raise

for path in generated_paths:
    if Path(path).suffix.lower() in {".srt", ".vtt"}:
        normalize_subtitle(path)
print(f"\n✅ Saved {len(generated_paths)} file(s) persistently in {output_dir}")


## 5️⃣ 💬 Translate with DeepL *(optional)*

This step sends transcript text—not audio—to DeepL after you explicitly run it. Add `DEEPL_API_KEY` in Colab Secrets first. Translation progress is checkpointed in Drive after every batch.

Translated files are saved in the Step 4 output folder.


In [ ]:
#@title 💬 Step 5 — Translate with DeepL { display-mode: "form" }
import hashlib
import json
import math
import os
import tempfile
import time
from pathlib import Path
import deepl
from google.colab import userdata

#@markdown Transcript text is sent to DeepL only when this cell is run.
deepl_target_language = "English (American)" #@param ["Bulgarian", "Chinese (simplified)", "Czech", "Danish", "Dutch", "English (American)", "English (British)", "Estonian", "Finnish", "French", "German", "Greek", "Hungarian", "Indonesian", "Italian", "Japanese", "Korean", "Latvian", "Lithuanian", "Norwegian", "Polish", "Portuguese (Brazilian)", "Portuguese (European)", "Romanian", "Russian", "Slovak", "Slovenian", "Spanish", "Swedish", "Turkish", "Ukrainian"]
deepl_formality = "default" #@param ["default", "formal", "informal"]
share_context = True #@param {type:"boolean"}

try:
    deepl_api_key = userdata.get("DEEPL_API_KEY") or ""
except Exception:
    deepl_api_key = ""
if not deepl_api_key:
    raise RuntimeError("Add DEEPL_API_KEY in Colab Secrets and enable notebook access")

if "results" not in globals() or not results:
    checkpoint_payload = load_results_checkpoint(CHECKPOINT_PATH)
    results = checkpoint_payload["results"]
    task = checkpoint_payload.get("task", "transcribe")
if "write_result" not in globals() or "output_formats" not in globals():
    raise RuntimeError("Run Step 4 once before Step 5 to choose output settings")

DEEPL_CHECKPOINT = os.path.join(output_dir, "_last_deepl_results.json")

def atomic_deepl_checkpoint(data):
    temporary = None
    try:
        with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=output_dir,
                                         suffix=".tmp", delete=False) as handle:
            temporary = handle.name
            json.dump({"schema_version": 1, "target_code": target_code,
                       "source_signature": source_signature, "results": data},
                      handle, ensure_ascii=False, indent=2)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, DEEPL_CHECKPOINT)
    except Exception:
        if temporary:
            Path(temporary).unlink(missing_ok=True)
        raise

def valid_resume_prefix(saved_segments, source_segments):
    if not isinstance(saved_segments, list) or len(saved_segments) > len(source_segments):
        return False
    for saved, source in zip(saved_segments, source_segments):
        if saved.get("id") != source.get("id"):
            return False
        if float(saved.get("start", -1)) != float(source.get("start", -2)):
            return False
        if float(saved.get("end", -1)) != float(source.get("end", -2)):
            return False
    return True

def translate_with_retry(translator, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return translator.translate_text(**kwargs)
        except Exception as error:
            retryable = any(word in str(error).lower()
                            for word in ("timed out", "timeout", "temporarily", "429"))
            if retryable and attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"⚠️ Temporary DeepL error; retrying in {wait}s…")
                time.sleep(wait)
            else:
                raise

translator = deepl.Translator(deepl_api_key)
targets = translator.get_target_languages()
target = next((item for item in targets if item.name == deepl_target_language), None)
if not target:
    raise ValueError(f"DeepL target is unavailable: {deepl_target_language}")
target_code = target.code.upper()
formality = None if deepl_formality == "default" else (
    "prefer_more" if deepl_formality == "formal" else "prefer_less")
source_codes = {item.code.upper() for item in translator.get_source_languages()}
source_signature = hashlib.sha256(
    json.dumps(results, ensure_ascii=False, sort_keys=True, default=str).encode("utf-8")
).hexdigest()
translated_results = {}
if os.path.isfile(DEEPL_CHECKPOINT):
    try:
        with open(DEEPL_CHECKPOINT, encoding="utf-8") as handle:
            saved_translation = json.load(handle)
        compatible = (
            saved_translation.get("schema_version") == 1
            and saved_translation.get("target_code") == target_code
            and saved_translation.get("source_signature") == source_signature
        )
        if compatible:
            translated_results = saved_translation.get("results", {})
            print(f"♻️ Loaded compatible DeepL checkpoint: {DEEPL_CHECKPOINT}")
        else:
            print("ℹ️ Existing DeepL checkpoint belongs to different inputs/settings; starting fresh.")
    except (OSError, ValueError, TypeError) as error:
        print(f"⚠️ Ignoring invalid DeepL checkpoint: {error}")

print("🔒 Sending transcript text to DeepL. Audio is not uploaded.")
for audio_path, result in results.items():
    existing = translated_results.get(audio_path, {})
    if existing.get("translation_status") == "complete":
        print(f"♻️ Reusing completed DeepL translation: {audio_path}")
        continue
    source_code = str(result.get("language", "")).upper()
    if len(source_code) > 2:
        import whisper as whisper_tokenizer
        source_code = whisper_tokenizer.tokenizer.TO_LANGUAGE_CODE.get(
            source_code.lower(), "").upper()
    if source_code == target_code.split("-")[0]:
        print(f"Skipping {audio_path}: already in target language")
        continue
    if task == "transcribe" and source_code not in source_codes:
        print(f"Skipping {audio_path}: DeepL does not support source {source_code}")
        continue

    segments = result.get("segments", [])
    saved_segments = existing.get("segments", [])
    if existing.get("translation_status") == "in_progress" and not valid_resume_prefix(saved_segments, segments):
        print(f"⚠️ Discarding inconsistent DeepL checkpoint entry: {audio_path}")
        existing = {}
    translated = existing if existing.get("translation_status") == "in_progress" else {
        "text": "", "segments": [], "language": deepl_target_language,
        "translation_status": "in_progress"}
    translated_results[audio_path] = translated
    batch_size = 50
    completed_segments = min(len(translated.get("segments", [])), len(segments))
    for start in range(completed_segments, len(segments), batch_size):
        batch = segments[start:start + batch_size]
        texts = [segment.get("text", "") for segment in batch]
        context = "\n".join(texts) if share_context else None
        responses = translate_with_retry(
            translator, text=texts,
            source_lang=source_code if task == "transcribe" else None,
            target_lang=target_code, formality=formality,
            context=context, split_sentences="nonewlines",
        )
        if not isinstance(responses, list):
            responses = [responses]
        if len(responses) != len(batch):
            raise RuntimeError(f"DeepL returned {len(responses)} results for {len(batch)} segments")
        for segment, response in zip(batch, responses):
            translated_text = response.text.strip()
            translated["segments"].append({
                "id": segment.get("id"), "start": segment["start"],
                "end": segment["end"], "text": translated_text,
            })
        translated["text"] = "\n".join(item["text"] for item in translated["segments"])
        atomic_deepl_checkpoint(translated_results)
        print(f"Translated {min(start + batch_size, len(segments))}/{len(segments)} segments")
        time.sleep(1)
    translated["translation_status"] = "complete"
    atomic_deepl_checkpoint(translated_results)

name_map = output_name_map(list(translated_results))
for audio_path, translated in translated_results.items():
    if translated.get("translation_status") != "complete":
        continue
    base = f"{name_map[audio_path]}-{target_code.lower()}"
    base = reserve_output_name(base, output_formats, output_dir,
                               overwrite=overwrite_existing)
    for output_format in output_formats:
        path = write_result(translated, output_format, base)
        if Path(path).suffix.lower() in {".srt", ".vtt"}:
            normalize_subtitle(path)
        print("✅", path)
print(f"\n✅ DeepL outputs saved in {output_dir}")
